# Étape 2 — Optimisation (balayage de configurations, métrique **officielle**)

Baseline obtenue (`01_baseline_bpe_10k.ipynb`) : **score officiel = 2.059977**, guardrails EN/FR **PASS**.

Ce notebook cherche à **faire baisser ce score** en testant plusieurs configurations, en utilisant
**exactement la métrique officielle du challenge** (lue dans le dépôt officiel
[`airf-multilingual-tokenizer-challenge`](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge), `competition/metrics.py` + `competition/constants.py`).

## Règles officielles (vérifiées dans le code du challenge)

| Élément | Valeur officielle |
|---|---|
| Mot | `len(text.split())` (espaces blancs) |
| Fertility | `tokens / words` |
| Pénalité UNK | `100 × (unk_tokens / words)` |
| Score par langue | `fertility + 100 × unk_rate` |
| Score final | moyenne de **ha, sw, yo, am** |
| Guardrail EN/FR | `budget = 1.15 × moyenne(fertility brute ha,sw,yo,am)` ; échec si `fertility(en)` **ou** `fertility(fr)` > budget |
| Vocabulaire max | 10 000 (`get_vocab_size(with_added_tokens=True)`) |
| Taille max du fichier | 20 MiB |
| Version `tokenizers` | **`0.22.1` exactement** (sinon `compatible_version` échoue) |
| Données | **uniquement** le `train` fourni par le challenge |

> ⚠️ **Point stratégique majeur** : le budget du guardrail est **relatif** — il vaut 1,15 × la moyenne
> des langues africaines **de votre propre tokenizer**. Donc si l'on améliore beaucoup ha/sw/yo/am
> sans améliorer en/fr, le budget baisse et le guardrail peut **casser** (soumission rejetée).
> Chaque configuration est donc évaluée **avec son verdict de guardrail**.

## Axes testés (7–8 configurations)

1. **`b1-baseline`** — référence : `BPE + NFC + Whitespace`, vocab 10k, `min_frequency=2` (doit redonner ≈ 2.0600).
2. **`c2-wssplit-mf2`** — pré-tokeniseur `WhitespaceSplit` : la ponctuation **reste collée** au mot
   (`shared.` = 1 pré-token au lieu de 2). Le baseline isole `,` et `.` qui représentaient à eux seuls
   ~43 000 tokens sur la validation.
3. **`c3-wssplit-mf5`** — idem + `min_frequency=5` (moins de fusions rares gaspillées).
4. **`c4-wssplit-mf10-alpha`** — idem + `min_frequency=10` + **alphabet initial = tous les caractères du train**
   (supprime les `[UNK]` pour tout caractère vu à l'entraînement).
5. **`c5-bytelevel`** — `ByteLevel(use_regex=True)` + `BPE` : round-trip **sans perte** et **zéro `[UNK]`**,
   mais coût en octets pour l'éthiopien (3 octets/caractère) → à mesurer.
6. **`c6-unigram-alpha`** — modèle **Unigram** (souvent meilleur à vocabulaire fixe) + alphabet complet.
7. **`c7-...-amboost`** — comme c4 + **sur-échantillonnage de l'amharique (×2)** : l'amharique est la pire
   langue (fertility 2,49 / 130 `[UNK]`) ; on lui alloue plus de fusions.
8. **`c8-scoredboost`** — sur-échantillonnage des **4 langues notées** (×2) : teste la limite du guardrail
   (baisse le score mais peut faire échouer EN/FR — c'est justement ce que l'on veut mesurer).

À la fin : tableau classé, verdict guardrail, **sauvegarde du meilleur**, rapport
`reports/optimization_sweep.{json,md}`, dossier de soumission `submissions/<slug>/` et
exécution du **checker officiel** (`starter/utils.py`).

## 1. Installation (versions officielles)

`tokenizers==0.22.1` est **imposé** par le challenge : le checker officiel vérifie l'égalité exacte
de version (`SUPPORTED_TOKENIZERS_VERSION = "0.22.1"`). On épingle donc cette version.

In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy
import tokenizers
print("tokenizers:", tokenizers.__version__, "(attendu 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Installer tokenizers==0.22.1 (exigence officielle)"

## 2. Constantes, métrique officielle et guardrail

Le code ci-dessous reproduit **exactement** la métrique officielle
(`competition/metrics.py`) et les constantes (`competition/constants.py`).

In [ ]:
# =============================================================================
# 2. Constantes + métrique OFFICIELLES (compétition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Constantes officielles (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Dataset officiel ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Paramètres du balayage (modifiables) ----------------------------------
VOCAB_SIZE = 10_000            # imposé par le challenge
MAX_TRAIN_DOCS = None          # None = tout le train (240 000) ; ex. 60_000 pour un pré-balayage rapide
BASELINE_REFERENCE_SCORE = 2.059977   # score officiel obtenu par 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- MÉTRIQUE OFFICIELLE ---------------------------------------------------
def count_words(text: str) -> int:
    """Mots = séparés par des espaces blancs (comme l'évaluateur officiel)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id émis pour un texte non représentable (comme l'évaluateur officiel)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = liste de (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Score officiel = moyenne des langues notées (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"langues notées manquantes : {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x moyenne(fertility brute des langues notées)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


def validate_tokenizer_file(path):
    """Contrôles officiels de validation (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("fichier > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulaire {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("une langue ne produit aucun token")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("un décodage est vide")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Métrique officielle chargée. Vocab max:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

## 3. Données : chargement + préparation

Entraînement sur `train` uniquement, évaluation sur `validation` uniquement (comme le baseline).

In [ ]:
# =============================================================================
# 3. Chargement du dataset officiel + préparation des textes
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nEntraînement :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation   :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "train inattendu"
assert len(val_rows) == 24_000, "validation inattendue"

## 4. Configurations candidates

Chaque configuration est décrite par un dict : modèle, normaliseur, pré-tokeniseur, `min_frequency`,
alphabet initial (complet / bytes) et sur-échantillonnage éventuel.

In [ ]:
# =============================================================================
# 4. Définition des configurations candidates
# =============================================================================
from tokenizers.models import BPE, Unigram
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPre, Whitespace, WhitespaceSplit
from tokenizers.decoders import ByteLevel as ByteLevelDec
from tokenizers.trainers import BpeTrainer, UnigramTrainer

CONFIGS = [
    dict(name="b1-baseline", model="bpe", pre="whitespace", min_freq=2,
         alphabet=False, boost={},
         note="référence : BPE+NFC+Whitespace, mf=2 (doit redonner ~2.0600)"),
    dict(name="c2-wssplit-mf2", model="bpe", pre="whitespace_split", min_freq=2,
         alphabet=False, boost={},
         note="ponctuation collée au mot (WhitespaceSplit)"),
    dict(name="c3-wssplit-mf5", model="bpe", pre="whitespace_split", min_freq=5,
         alphabet=False, boost={},
         note="WhitespaceSplit + min_frequency=5"),
    dict(name="c4-wssplit-mf10-alpha", model="bpe", pre="whitespace_split", min_freq=10,
         alphabet=True, boost={},
         note="WhitespaceSplit + mf=10 + alphabet complet (supprime les UNK connus)"),
    dict(name="c5-bytelevel", model="bpe", pre="byte_level", min_freq=2,
         alphabet="bytes", boost={},
         note="ByteLevel(use_regex) : round-trip sans perte, zero UNK"),
    dict(name="c6-unigram-alpha", model="unigram", pre="whitespace_split", min_freq=2,
         alphabet=True, boost={},
         note="modele Unigram + alphabet complet"),
    dict(name="c7-wssplit-mf10-alpha-amboost", model="bpe", pre="whitespace_split", min_freq=10,
         alphabet=True, boost={"am": 2},
         note="comme c4 + amharique sur-echantillonne x2"),
    dict(name="c8-scoredboost", model="bpe", pre="whitespace_split", min_freq=5,
         alphabet=True, boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="sur-echantillonnage des 4 langues notees (teste la limite du guardrail)"),
]

# Alfabet : caractères du train (>=1 occurrence) pour 'alphabet=True'
if any(c["alphabet"] is True for c in CONFIGS):
    train_chars = set()
    for texts in train_by_lang.values():
        for t in texts:
            train_chars.update(ch for ch in t if not ch.isspace())
    train_alphabet = sorted(train_chars)
    print(f"Alphabet du train : {len(train_alphabet):,} caractères distincts (hors espaces)")
else:
    train_alphabet = []

# Tirage : permet de lancer un sous-ensemble  ->  RUN_CONFIGS = {"c4-wssplit-mf10-alpha"}
RUN_CONFIGS = None            # None = toutes les configurations
selected = [c for c in CONFIGS if RUN_CONFIGS is None or c["name"] in RUN_CONFIGS]
print(f"\n{len(selected)} configuration(s) à entraîner :")
for c in selected:
    print(f"  - {c['name']:32s} {c['note']}")

## 5. Entraînement + évaluation de chaque configuration

Pour chaque configuration : entraînement sur `train`, puis évaluation **sur `validation`** avec la
métrique officielle (score, fertility par langue, UNK, **verdict guardrail**).

⏱️ Durée indicative dans Colab : quelques minutes par configuration (8 configurations ≈ 20–40 min).
Utiliser `MAX_TRAIN_DOCS = 60_000` dans la cellule 2 pour un pré-balayage rapide, puis relancer les
2–3 meilleures en données complètes.

In [ ]:
# =============================================================================
# 5. Balayage : entraînement + évaluation officielle
# =============================================================================
def corpus_iterator(train_by_lang, boost=None, log_every=50_000):
    """Itère les textes multilingues en round-robin (équilibré) avec sur-échantillonnage."""
    boost = boost or {}
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if log_every and i % log_every == 0:
                    print(f"    ... {i:,} textes fournis")


def build_tokenizer(cfg):
    """Construit et entraîne un tokenizer selon la configuration (utilise train seulement)."""
    if cfg["model"] == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
    else:
        tokenizer = Tokenizer(Unigram())
    tokenizer.normalizer = NFC()

    if cfg["pre"] == "whitespace":
        tokenizer.pre_tokenizer = Whitespace()
    elif cfg["pre"] == "whitespace_split":
        tokenizer.pre_tokenizer = WhitespaceSplit()
    elif cfg["pre"] == "byte_level":
        tokenizer.pre_tokenizer = ByteLevelPre(add_prefix_space=False, use_regex=True)
        tokenizer.decoder = ByteLevelDec()

    if cfg["alphabet"] == "bytes":
        alphabet = ByteLevelPre.alphabet()
    elif cfg["alphabet"] is True:
        alphabet = train_alphabet
    else:
        alphabet = None

    if cfg["model"] == "bpe":
        kwargs = dict(vocab_size=VOCAB_SIZE, min_frequency=cfg["min_freq"], special_tokens=["[UNK]"])
        if alphabet:
            kwargs["initial_alphabet"] = alphabet
        trainer = BpeTrainer(**kwargs)
    else:
        kwargs = dict(vocab_size=VOCAB_SIZE, special_tokens=["[UNK]"], unk_token="[UNK]")
        if alphabet:
            kwargs["initial_alphabet"] = alphabet
        trainer = UnigramTrainer(**kwargs)

    tokenizer.train_from_iterator(
        corpus_iterator(train_by_lang, cfg.get("boost")), trainer=trainer)
    return tokenizer


results = []
for n, cfg in enumerate(selected, start=1):
    print(f"\n{'='*84}\n[{n}/{len(selected)}] {cfg['name']} — {cfg['note']}\n{'='*84}")
    t0 = time.time()
    tok = build_tokenizer(cfg)
    train_seconds = time.time() - t0

    fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tok)
    score = competition_score(fertility, unk_rate)
    raw, budget, breaches = guardrail(fertility)
    penalised = penalised_scores(fertility, unk_rate)
    vocab_size = tok.get_vocab_size(with_added_tokens=True)

    results.append(dict(
        name=cfg["name"], note=cfg["note"], config=cfg,
        vocab_size=vocab_size, train_seconds=train_seconds,
        score=score, fertility=fertility, unk_rate=unk_rate, penalised=penalised,
        tokens=tokens, words=words, unk_total=unk_total, lossy_rows=lossy,
        raw_scored=raw, guardrail_budget=budget, guardrail_breaches=breaches,
        guardrail_pass=not breaches, tokenizer=tok,
    ))
    flag = "GUARDRAIL OK" if not breaches else f"GUARDRAIL FAIL {breaches}"
    print(f"  score={score:.4f} | vocab={vocab_size} | UNK={unk_total} | lossy={lossy:,} | "
          f"{flag} | {train_seconds/60:.1f} min")

print("\nBalayage terminé.")

## 6. Résultats classés

Tableau trié par score officiel (plus bas = meilleur), avec le verdict guardrail de chaque configuration.

In [ ]:
# =============================================================================
# 6. Tableau comparatif + classement
# =============================================================================
rows = []
for r in sorted(results, key=lambda x: x["score"]):
    rows.append({
        "config": r["name"],
        "Score ↓": round(r["score"], 4),
        "Guardrail": "PASS" if r["guardrail_pass"] else "FAIL",
        "Hausa": round(r["penalised"]["ha"], 4),
        "Swahili": round(r["penalised"]["sw"], 4),
        "Yoruba": round(r["penalised"]["yo"], 4),
        "Amharic": round(r["penalised"]["am"], 4),
        "UNK": r["unk_total"],
        "lossy": f"{r['lossy_rows']:,}",
        "vocab": r["vocab_size"],
        "en": round(r["fertility"]["en"], 4),
        "fr": round(r["fertility"]["fr"], 4),
        "budget": round(r["guardrail_budget"], 4),
    })
sweep_df = pd.DataFrame(rows)
print("Score officiel = moyenne des scores (ha, sw, yo, am). Lower is better.")
print("Guardrail = fertility(en) et fertility(fr) <= 1.15 x moyenne brute des 4 langues notées.\n")
print(sweep_df.to_string(index=False))

valid_results = [r for r in results if r["guardrail_pass"]]
best = min(valid_results, key=lambda r: r["score"]) if valid_results else None

print("\n--- Comparaison à la baseline (2.059977) ---")
baseline_result = next((r for r in results if r["name"] == "b1-baseline"), None)
if baseline_result:
    delta_check = baseline_result["score"] - BASELINE_REFERENCE_SCORE
    print(f"b1-baseline reproduit {baseline_result['score']:.4f} "
          f"(référence {BASELINE_REFERENCE_SCORE:.4f}, écart {delta_check:+.4f})")
if best:
    gain = BASELINE_REFERENCE_SCORE - best["score"]
    print(f"\nMEILLEURE CONFIGURATION ÉLIGIBLE : {best['name']}")
    print(f"  score {best['score']:.4f}  (gain vs baseline : {gain:+.4f} soit {100*gain/BASELINE_REFERENCE_SCORE:.1f} %)")
    print(f"  guardrail : budget {best['guardrail_budget']:.4f} | en {best['fertility']['en']:.4f} | fr {best['fertility']['fr']:.4f}")
    for l in LANGUAGES:
        print(f"    {l}: fertility {best['fertility'][l]:.4f} | unk_rate {best['unk_rate'][l]:.6f} | score {best['penalised'][l]:.4f}")
else:
    print("Aucune configuration ne passe le guardrail : revoir les configurations.")

## 7. Analyse automatique : quel levier a fonctionné ?

Comparaison des axes testés pour comprendre **pourquoi** une configuration gagne.

In [ ]:
# =============================================================================
# 7. Analyse des leviers
# =============================================================================
base = next((r for r in results if r["name"] == "b1-baseline"), None)
print("Effet des leviers (score relatif à la baseline) :\n")
for r in sorted(results, key=lambda x: x["score"]):
    rel = "" if base is None else f"{r['score'] - base['score']:+.4f}"
    print(f"  {r['name']:34s} {r['score']:.4f}  ({rel})")

print("\nLecture des UNK :")
for r in results:
    print(f"  {r['name']:34s} UNK total={r['unk_total']:>6} | " +
          " ".join(f"{l}:{r['unk_rate'][l]*100:.3f}%" for l in LANGUAGES))

print("\nLecture de la fertility par langue (brute) :")
print(f"  {'config':34s} " + " ".join(f"{l:>8s}" for l in LANGUAGES))
for r in results:
    print(f"  {r['name']:34s} " + " ".join(f"{r['fertility'][l]:>8.4f}" for l in LANGUAGES))

## 8. Sauvegarde du meilleur modèle + rapports

Le tokenizer gagnant est écrit dans `models/optimized_<config>/tokenizer.json` et les rapports dans
`reports/optimization_sweep.{json,md}`.

In [ ]:
# =============================================================================
# 8. Sauvegarde du meilleur + rapports JSON/Markdown
# =============================================================================
best_model_dir = MODEL_DIR / f"optimized_{best['name']}"
best_model_dir.mkdir(parents=True, exist_ok=True)
best_tokenizer_path = best_model_dir / "tokenizer.json"
best["tokenizer"].save(str(best_tokenizer_path))
print("Meilleur tokenizer sauvegardé :", best_tokenizer_path,
      f"({best_tokenizer_path.stat().st_size:,} octets)")

# Copie "candidate" à la racine pour le checker officiel
candidate_path = OUTPUT_ROOT / "tokenizer.json"
shutil.copy2(best_tokenizer_path, candidate_path)
print("Candidat pour le checker :", candidate_path)

report = {
    "experiment": "optimization_sweep",
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_docs_used": {l: len(v) for l, v in train_by_lang.items()},
                "validation_rows": len(val_rows)},
    "official_rules": {
        "scored_languages": list(SCORED_LANGUAGES),
        "context_languages": list(CONTEXT_LANGUAGES),
        "guardrail_ratio": CONTEXT_FERTILITY_RATIO,
        "unknown_penalty": UNKNOWN_PENALTY,
        "max_vocab_size": MAX_VOCAB_SIZE,
        "required_tokenizers_version": REQUIRED_TOKENIZERS_VERSION,
        "word_definition": "len(text.split())",
    },
    "settings": {"vocab_size": VOCAB_SIZE, "max_train_docs": MAX_TRAIN_DOCS},
    "baseline_reference_score": BASELINE_REFERENCE_SCORE,
    "results": [
        {k: v for k, v in r.items() if k not in ("tokenizer",)}
        for r in sorted(results, key=lambda x: x["score"])
    ],
    "best": {"name": best["name"], "score": best["score"], "config": best["config"],
             "guardrail_pass": best["guardrail_pass"], "model_path": str(best_tokenizer_path)},
    "environment": {"tokenizers": __import__("tokenizers").__version__,
                    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
}
(REPORT_DIR / "optimization_sweep.json").write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                                   encoding="utf-8")

# --- Markdown ---------------------------------------------------------------
md = ["# Étape 2 — Balayage d'optimisation (métrique officielle)", "",
      f"*Dataset `{DATASET_NAME}` @ `{DATASET_REVISION}` — train utilisé : "
      f"{sum(len(v) for v in train_by_lang.values()):,} textes, validation : {len(val_rows):,} lignes.*", "",
      "## Classement", "",
      "| Config | Score ↓ | Guardrail | Hausa | Swahili | Yoruba | Amharic | UNK | lossy | vocab | en | fr | budget |",
      "|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | {r['score']:.4f} | {'PASS' if r['guardrail_pass'] else 'FAIL'} | "
              f"{r['penalised']['ha']:.4f} | {r['penalised']['sw']:.4f} | {r['penalised']['yo']:.4f} | "
              f"{r['penalised']['am']:.4f} | {r['unk_total']} | {r['lossy_rows']:,} | {r['vocab_size']} | "
              f"{r['fertility']['en']:.4f} | {r['fertility']['fr']:.4f} | {r['guardrail_budget']:.4f} |")
md += ["", "## Fertility brute par langue", "",
       "| Config | " + " | ".join(LANGUAGES) + " |", "|---|" + "---:|" * len(LANGUAGES)]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | " + " | ".join(f"{r['fertility'][l]:.4f}" for l in LANGUAGES) + " |")
md += ["", "## Décision", "",
       f"- Baseline de référence : **{BASELINE_REFERENCE_SCORE:.4f}**",
       f"- **Meilleure configuration (guardrail OK) : `{best['name']}` — score {best['score']:.4f}** "
       f"({BASELINE_REFERENCE_SCORE - best['score']:+.4f})",
       f"- Guardrail : budget {best['guardrail_budget']:.4f}, en {best['fertility']['en']:.4f}, "
       f"fr {best['fertility']['fr']:.4f} → {'PASS' if best['guardrail_pass'] else 'FAIL'}",
       f"- Modèle : `{best_tokenizer_path}`", "",
       "## Configurations testées", ""]
for r in results:
    md.append(f"- `{r['name']}` — {r['note']} — score {r['score']:.4f} — "
              f"guardrail {'PASS' if r['guardrail_pass'] else 'FAIL'}")
md.append("")
(REPORT_DIR / "optimization_sweep.md").write_text("\n".join(md), encoding="utf-8")
print("Rapports :", REPORT_DIR / "optimization_sweep.json", "|", REPORT_DIR / "optimization_sweep.md")

## 9. Vérification avec le **checker officiel** du challenge

On télécharge `starter/utils.py` du dépôt officiel et on exécute `profile_submission` sur le
tokenizer gagnant, avec le split de validation comme données. C'est **le même code** que celui
utilisé pour valider les soumissions.

In [ ]:
# =============================================================================
# 9. Checker officiel (starter/utils.py du dépôt du challenge)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("utils.py officiel téléchargé")
    except Exception as exc:
        print("Téléchargement impossible :", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Checker officiel indisponible — utilisation des contrôles intégrés.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Contrôles intégrés complémentaires (équivalents à competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nContrôles officiels intégrés :", checks["checks"])
print("Erreurs :", checks["errors"] or "aucune")
print("Langues non lossless (round-trip) :", checks["lossy_languages"] or "aucune (lossless)")

## 10. Dossier de soumission `submissions/<slug>/`

Génère le dossier attendu par le challenge (voir `CONTRIBUTING.md` du dépôt officiel) :
`tokenizer.json` (obligatoire), `metadata.yml` (obligatoire), `README.md` (approche).

⚠️ **Remplissez `TEAM_NAME`, `MEMBERS` et `AFFILIATION`** ci-dessous : ils sont publics dans la
soumission. Le slug du dossier doit être en minuscules kebab-case (ex. `maick-code`).

In [ ]:
# =============================================================================
# 10. Génération du dossier de soumission
# =============================================================================
TEAM_NAME = "À REMPLIR — nom d'équipe"
MEMBERS = ["À REMPLIR — membre 1"]           # au plus 6 membres
AFFILIATION = ""                              # optionnel (<= 120 caractères)
SLUG = "a-remplir"                            # minuscules kebab-case, ex. "maick-code"

assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", SLUG), "slug invalide (minuscules kebab-case)"
assert len(MEMBERS) <= 6, "6 membres maximum"

submission_dir = SUBMISSIONS_DIR / SLUG
submission_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(best_tokenizer_path, submission_dir / "tokenizer.json")

approach = (f"BPE/Unigram tokenizer ({best['name']}) trained on the official competition train split only. "
            f"Official validation score {best['score']:.4f} (baseline BPE 10k: {BASELINE_REFERENCE_SCORE:.4f}). "
            f"English/French guardrail PASS.")[:240]
metadata = {
    "team": TEAM_NAME,
    "members": MEMBERS,
    "affiliation": AFFILIATION[:120],
    "approach": approach,
}
try:
    import yaml
    (submission_dir / "metadata.yml").write_text(
        yaml.safe_dump(metadata, allow_unicode=True, sort_keys=False), encoding="utf-8")
    print("metadata.yml écrit (format YAML).")
except ImportError:
    lines = [f"team: {TEAM_NAME}", "members:"]
    lines += [f"  - {m}" for m in MEMBERS]
    lines += [f"affiliation: {AFFILIATION}", f"approach: {approach}"]
    (submission_dir / "metadata.yml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("metadata.yml écrit (YAML manuel).")

(submission_dir / "README.md").write_text(
    f"# {TEAM_NAME}\n\nApproche : `{best['name']}`\n\n{best['note']}\n\n"
    f"- Score officiel (validation) : **{best['score']:.4f}** (baseline {BASELINE_REFERENCE_SCORE:.4f})\n"
    f"- Guardrail EN/FR : **PASS** (budget {best['guardrail_budget']:.4f}, "
    f"en {best['fertility']['en']:.4f}, fr {best['fertility']['fr']:.4f})\n"
    f"- Vocabulaire : {best['vocab_size']:,} / 10 000\n"
    f"- UNK sur validation : {best['unk_total']}\n"
    f"- Données : split `train` officiel uniquement, aucun corpus externe.\n",
    encoding="utf-8")

print("\nDossier de soumission :", submission_dir)
for p in sorted(submission_dir.iterdir()):
    print(f"  {p.name} ({p.stat().st_size:,} octets)")
print("\nFichiers autorisés par le challenge : tokenizer.json, metadata.yml, notebook.ipynb, README.md")

## 11. Prochaines étapes

1. **Remplir** `TEAM_NAME`, `MEMBERS`, `SLUG` (cellule 10) puis relancer la cellule 10.
2. **Copier** `notebook.ipynb` dans le dossier de soumission (obligatoire avant la date limite) :
   c'est ce notebook-ci.
3. **Publier sur GitHub** :
   - *fork* de `aims-ai-research-foundations/airf-multilingual-tokenizer-challenge` ;
   - branche nommée exactement **`submission`** ;
   - `submissions/<slug>/` ajouté, puis *Pull Request* vers le dépôt officiel.
4. **PR** : cocher la checklist du template et décrire l'approche.

Rappels de contrainte (vérifiés dans le code officiel) :
- une PR ne doit modifier **que** `submissions/<slug>/` (3 niveaux de chemin, un seul slug) ;
- le dossier ne peut contenir que `tokenizer.json`, `metadata.yml`, `notebook.ipynb`, `README.md` ;
- pas de symlink ; `metadata.yml` ≤ 16 KiB ; `tokenizer.json` ≤ 20 MiB ; vocab ≤ 10 000 ;
- **`tokenizers==0.22.1`** (le workflow du challenge installe cette version via `uv.lock`) ;
- évaluation : temps ≤ 5× celui de la baseline (les tie-breaks se font sur la vitesse).

In [ ]:
# =============================================================================
# 11. Récapitulatif final
# =============================================================================
print("Balayage d'optimisation terminé.\n")
print(f"Baseline de référence : {BASELINE_REFERENCE_SCORE:.4f}")
print(f"Meilleure configuration éligible : {best['name']} -> {best['score']:.4f} "
      f"({BASELINE_REFERENCE_SCORE - best['score']:+.4f})")
print(f"Guardrail : {'PASS' if best['guardrail_pass'] else 'FAIL'}")
print()
print("Artefacts :")
print(f"  - {best_tokenizer_path}")
print(f"  - {REPORT_DIR / 'optimization_sweep.json'}")
print(f"  - {REPORT_DIR / 'optimization_sweep.md'}")
print(f"  - {submission_dir} (soumission)")
if official_report is not None:
    print("\nChecker officiel :", "READY FOR SUBMISSION" if official_report.get("valid")
          else f"NOT READY -> {official_report.get('errors')}")